# Bonus 04 — LiteLLM into Google ADK

Google ADK is often introduced with Gemini, while many organizations already standardize on another model provider. This lab separates two decisions that tutorials often blur together:

1. **LiteLLM** translates a common call shape into a provider-specific model request.
2. **Google ADK** supplies the agent runtime: tools, sessions, events, callbacks, and application state.

You will first call the course's OpenAI model through LiteLLM. Then you will pass that same adapter into an ADK refund agent and inspect every event around one tool call.

By the end, you should be able to explain what each library owns, what it does not own, and why a framework does not remove your application's policy responsibilities.

This optional lab uses its own locked environment. It does not change modules 00–17 or the core course environment.

## 1. Learn — two layers, two jobs

```mermaid
flowchart LR
    APP["Your Python application"] --> L["LiteLLM completion interface"]
    L --> O["OpenAI API"]
    APP --> A["ADK LlmAgent"]
    A --> M["ADK LiteLlm model adapter"]
    M --> L
    A <--> T["Python policy tool"]
    R["ADK Runner"] --> A
    S["SessionService: events + state"] <--> R
    R --> E["Event stream to application"]
```

LiteLLM is below the agent loop. It standardizes how code talks to a model provider. ADK is around the loop. It decides how an agent, tools, state, and events participate in one run.

Neither layer decides whether a refund is allowed. That rule remains deterministic application code.

### Three surfaces that share a name

| Surface | What it is | Used in this lab? |
|---|---|---|
| LiteLLM Python SDK | An in-process translation layer with an OpenAI-compatible response shape | Yes |
| LiteLLM Proxy | A separately deployed gateway for routing, budgets, keys, fallbacks, and centralized policy | No |
| Google ADK | A code-first agent and workflow runtime with tools, sessions, callbacks, events, evaluation, and deployment integrations | Yes |

A common interface reduces integration work. It does **not** make providers identical. Model names, supported parameters, tool behavior, structured-output behavior, latency, safety controls, and pricing can still differ. Test the capabilities you depend on.

This course continues to use OpenAI only. The provider prefix in `openai/<model>` teaches LiteLLM's routing convention; it is not an invitation to add providers without an operational reason.

### Why this lab has a separate environment

Google recommends a virtual environment for ADK. Here isolation is especially important: this lab resolves more than one hundred packages and selects a newer OpenAI SDK than the core course. Installing it into the shared room environment could break earlier modules.

From this lab directory, the instructor prepared the environment with:

```bash
uv sync --locked
```

In VS Code, select `bonus/04_litellm_google_adk/.venv/bin/python` as the notebook kernel. The notebook never installs packages.

There is also a real supply-chain lesson. LiteLLM versions **1.82.7 and 1.82.8** were compromised on PyPI in March 2026. This lab pins a later reviewed release in `uv.lock`, keeps it away from the core environment, and forces LiteLLM to use its packaged model-cost map rather than fetching a mutable copy during import. Exact pins and a lockfile reduce exposure; they do not replace dependency review, secret rotation procedures, or artifact verification.

## 2. Do — meet LiteLLM before ADK

The setup cell searches upward for the repository `.env`, then sets LiteLLM's cost-map mode **before** importing LiteLLM. It prints versions and model names, never credentials.

In [ ]:
import os
from importlib.metadata import version
from pathlib import Path

from dotenv import load_dotenv

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists() and (path / "modules").exists()
)
load_dotenv(repo_root / ".env")

os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "true"

MODEL_DEFAULT = os.getenv("MODEL_DEFAULT")
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is missing from the repository .env"
assert MODEL_DEFAULT, "MODEL_DEFAULT is missing from the repository .env"

PRICE_INPUT = float(os.getenv("PRICE_INPUT_PER_MILLION", "0"))
PRICE_CACHED_INPUT = float(os.getenv("PRICE_CACHED_INPUT_PER_MILLION", "0"))
PRICE_OUTPUT = float(os.getenv("PRICE_OUTPUT_PER_MILLION", "0"))

print("google-adk:", version("google-adk"))
print("litellm:", version("litellm"))
print("openai:", version("openai"))
print("course model:", MODEL_DEFAULT)

### One normalized completion

LiteLLM's provider/model convention is `<provider>/<model>`. The prefix tells LiteLLM which adapter should translate the request. The provider still receives the real model name and performs the inference.

The course model family requires `max_completion_tokens`, no `temperature`, and `reasoning_effort="none"`. A common API does not remove those model-specific facts.

In [ ]:
from litellm import completion

litellm_model = f"openai/{MODEL_DEFAULT}"
direct_response = completion(
    model=litellm_model,
    messages=[
        {"role": "system", "content": "Answer as a concise enterprise support analyst."},
        {"role": "user", "content": "In one sentence, distinguish a model adapter from an agent runtime."},
    ],
    max_completion_tokens=100,
    reasoning_effort="none",
)

print(direct_response.choices[0].message.content)

### Inspect the envelope, not only the text

LiteLLM returns typed, OpenAI-compatible response objects across its supported providers. Application code can use a stable path to content and usage, while provider details remain visible.

In [ ]:
direct_message = direct_response.choices[0].message
direct_usage = direct_response.usage

print("response type:", type(direct_response).__name__)
print("message type:", type(direct_message).__name__)
print("provider model:", direct_response.model)
print("role:", direct_message.role)
print("finish reason:", direct_response.choices[0].finish_reason)
print("usage:", direct_usage.model_dump())

In [ ]:
direct_cached = direct_usage.prompt_tokens_details.cached_tokens or 0
direct_uncached = direct_usage.prompt_tokens - direct_cached
direct_cost = (
    direct_uncached * PRICE_INPUT
    + direct_cached * PRICE_CACHED_INPUT
    + direct_usage.completion_tokens * PRICE_OUTPUT
) / 1_000_000

print(f"direct call tokens: {direct_usage.total_tokens}")
print(f"estimated direct call cost: ${direct_cost:.6f}")

### Now give the adapter to Google ADK

ADK's Python integration wraps LiteLLM in `LiteLlm`. The wrapper translates ADK's model request and response objects into LiteLLM's interface.

The important ADK objects are:

| Object | Responsibility |
|---|---|
| `LlmAgent` | instructions, model, tools, callbacks, and output behavior |
| `Runner` | executes the loop and emits events |
| `SessionService` | stores each conversation's events and state |
| `ToolContext` | gives a tool controlled access to the current run and state |
| `Event` | records model requests, tool requests, tool results, state deltas, and final output |

The variable name `root_agent` is an ADK convention used by its command-line and web tooling. In this notebook we pass that same object directly to a `Runner`, so there is no hidden `.py` application.

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import ToolContext
from google.genai import types

### Keep policy deterministic

The model extracts facts from the request and chooses the tool. The Python function owns the refund rule. `ToolContext` is not exposed as a model-generated argument; ADK injects it when the application executes the tool.

The function also records its decision in session state. That state change will appear in the event stream and survive for later turns in this session.

In [ ]:
def check_refund_policy(
    amount_usd: float,
    days_since_purchase: int,
    opened: bool,
    tool_context: ToolContext,
) -> dict:
    """Apply the company's deterministic refund policy to a purchase."""
    if amount_usd > 500:
        decision = "human_review"
        reason = "Refunds above $500 require a person."
    elif days_since_purchase > 30 or opened:
        decision = "deny"
        reason = "Automatic refunds require an unopened item within 30 days."
    else:
        decision = "approve"
        reason = "The unopened item is within the 30-day window."

    tool_context.state["last_policy_decision"] = decision
    return {"decision": decision, "reason": reason}

### Callbacks observe a boundary

ADK callbacks can inspect or change execution at defined boundaries. These two callbacks only append local evidence before and after the tool. They are useful for teaching and debugging, but an in-memory list is not a durable enterprise audit store.

In [ ]:
audit_log = []

def before_tool(tool, args, tool_context):
    audit_log.append({
        "phase": "before_tool",
        "tool": tool.name,
        "args": dict(args),
    })

def after_tool(tool, args, tool_context, tool_response):
    audit_log.append({
        "phase": "after_tool",
        "tool": tool.name,
        "response": dict(tool_response),
    })

### Define the agent, then provide runtime services

`generate_content_config.max_output_tokens` is ADK's neutral field. Its LiteLLM connector maps that field to `max_completion_tokens` for the provider call. We still pin `reasoning_effort="none"` on the LiteLLM adapter because it is specific to this model family.

`output_key` tells ADK to save the final text into session state. The in-memory session service is appropriate for a lab, not for durable production recovery.

In [ ]:
APP_NAME = "refund_policy_lab"
USER_ID = "student-1"
SESSION_ID = "case-101"

root_agent = LlmAgent(
    name="refund_agent",
    model=LiteLlm(
        model=litellm_model,
        reasoning_effort="none",
    ),
    instruction=(
        "Always call check_refund_policy before answering a refund request. "
        "Report the tool's decision and reason. Never invent a different decision."
    ),
    tools=[check_refund_policy],
    before_tool_callback=before_tool,
    after_tool_callback=after_tool,
    output_key="last_answer",
    generate_content_config=types.GenerateContentConfig(max_output_tokens=140),
)

session_service = InMemorySessionService()
await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
    state={"customer_tier": "standard"},
)
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

print("agent:", root_agent.name)
print("model adapter:", type(root_agent.model).__name__)
print("session service:", type(session_service).__name__)

### Run one request and keep every event

`Runner.run_async` is an asynchronous event stream. The notebook collects it so we can inspect the protocol after the run instead of treating the final sentence as the only result.

In [ ]:
import warnings

user_message = types.Content(
    role="user",
    parts=[types.Part.from_text(
        text="A $129 unopened item was purchased 12 days ago. What should we do?"
    )],
)

events = []
final_text = None
with warnings.catch_warnings(record=True) as adk_notices:
    warnings.simplefilter("always")
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=user_message,
    ):
        events.append(event)
        if event.is_final_response() and event.content and event.content.parts:
            final_text = event.content.parts[0].text

for notice in adk_notices:
    print("ADK notice:", notice.message)
print(final_text)

## 3. Observe — the event stream is the execution record

A typical successful tool run produces three meaningful events:

1. the model emits a function name and JSON arguments;
2. the application executes the Python tool and records its response;
3. the model turns that response into the final answer.

The model did not execute Python. ADK's runner received a tool request and your application chose to execute the registered function.

In [ ]:
event_rows = []
for index, event in enumerate(events, start=1):
    calls = [call.name for call in event.get_function_calls()]
    responses = [response.name for response in event.get_function_responses()]
    usage = event.usage_metadata
    event_rows.append({
        "event": index,
        "author": event.author,
        "function_calls": calls,
        "function_responses": responses,
        "final": event.is_final_response(),
        "tokens": usage.total_token_count if usage else 0,
    })

for row in event_rows:
    print(row)

### Inspect the exact tool payloads

This is the same protocol lesson as the core course, now exposed through ADK objects. The first payload is the model's request. The second is the application's tool result.

In [ ]:
tool_call = next(
    call
    for event in events
    for call in event.get_function_calls()
)
tool_response = next(
    response
    for event in events
    for response in event.get_function_responses()
)

print("MODEL REQUESTED:")
print(tool_call.model_dump(exclude_none=True))
print("\nAPPLICATION RETURNED:")
print(tool_response.model_dump(exclude_none=True))

### Compare callbacks, state, and events

These are related but not interchangeable:

- callbacks ran application code at the tool boundary;
- state retained selected business values for this session;
- events retained the conversational and execution history.

A production design must choose durable service implementations and retention rules. `InMemorySessionService` loses everything when the process stops.

In [ ]:
completed_session = await session_service.get_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
)

print("callback evidence:")
for record in audit_log:
    print(record)

print("\nsession state:")
print(completed_session.state)
print("stored events:", len(completed_session.events))

### Aggregate model usage from events

The tool-response event has no model usage because Python produced it. The two model events carry usage: one to request the tool and one to write the final answer. Cost remains an application concern even when a framework runs the loop.

In [ ]:
prompt_tokens = sum(
    event.usage_metadata.prompt_token_count or 0
    for event in events
    if event.usage_metadata
)
cached_tokens = sum(
    event.usage_metadata.cached_content_token_count or 0
    for event in events
    if event.usage_metadata
)
output_tokens = sum(
    event.usage_metadata.candidates_token_count or 0
    for event in events
    if event.usage_metadata
)
agent_cost = (
    (prompt_tokens - cached_tokens) * PRICE_INPUT
    + cached_tokens * PRICE_CACHED_INPUT
    + output_tokens * PRICE_OUTPUT
) / 1_000_000

print({
    "prompt_tokens": prompt_tokens,
    "cached_tokens": cached_tokens,
    "output_tokens": output_tokens,
    "total_tokens": prompt_tokens + output_tokens,
})
print(f"estimated ADK run cost: ${agent_cost:.6f}")

### What the stack bought—and what it did not

| Concern | Owner in this lab | Still your responsibility |
|---|---|---|
| Provider request translation | LiteLLM | capability tests, credentials, budgets, provider policy |
| Agent/tool loop | ADK Runner | tool authorization, timeouts, failure policy |
| Conversation state | ADK SessionService | durable backend, tenancy, retention, deletion |
| Refund decision | Python tool | correct rules, tests, approvals, auditability |
| Execution evidence | ADK events and callbacks | secure storage, access control, monitoring |

Frameworks accelerate standard plumbing. They do not transfer accountability to the framework. If an organization wants more control, the core course has already shown the same loop in ordinary Python. If it wants faster development, ADK supplies reusable runtime surfaces without hiding the underlying tool protocol.

## 4. Challenge — build a contained high-value path

Create a second ADK agent for a `$725` unopened item purchased five days ago.

Acceptance criteria:

1. A deterministic tool returns `human_review` whenever `amount_usd > 500`.
2. The tool records that decision in `tool_context.state["last_policy_decision"]`.
3. The agent must call the tool before answering and must not replace its decision.
4. The session uses `InMemorySessionService` and starts with `customer_tier="priority"`.
5. Your code collects the event stream in `challenge_events` and the final text in `challenge_final_text`.
6. The verification cell must find exactly one policy call, one policy response, and the stored `human_review` state.

Use a new app name and session ID so the earlier run cannot satisfy the assertions accidentally.

In [ ]:
# Build your solution here. The separate solution notebook contains one answer.
# Suggested names used by the verification cell:
#   challenge_events
#   challenge_session
#   challenge_final_text

# def check_high_value_refund(...):
#     ...

# challenge_agent = LlmAgent(...)
# challenge_service = InMemorySessionService()
# ...

In [ ]:
challenge_calls = [
    call
    for event in challenge_events
    for call in event.get_function_calls()
]
challenge_responses = [
    response
    for event in challenge_events
    for response in event.get_function_responses()
]

assert len(challenge_calls) == 1
assert challenge_calls[0].name == "check_high_value_refund"
assert len(challenge_responses) == 1
assert challenge_responses[0].response["decision"] == "human_review"
assert challenge_session.state["last_policy_decision"] == "human_review"
assert "human_review" in challenge_final_text.lower()
print("Challenge passed: high-value refund was routed to human review.")

## Takeaway

LiteLLM and Google ADK solve different layers of the system. LiteLLM gives model integrations a common call and response shape. ADK gives agent execution a runner, tool boundary, session service, callbacks, and event stream.

The useful enterprise skill is not memorizing either constructor. It is preserving the boundaries:

- models propose;
- application code executes and enforces policy;
- frameworks coordinate;
- events make execution inspectable;
- durable systems still need deliberate security, state, cost, and dependency choices.